In [26]:
import pandas as pd
import re
import os

In [27]:
REGION = "ccr" # "full" or "ccr"

In [28]:
df = pd.read_csv(r"dataset\URA_merged_full_v1.csv")
print(df['Project Name'].nunique())
print(len(df.columns.tolist()))
for col in df.columns:
    print(col)

3239
367
Project Name
onemap_address
latitude
longitude
Street Name
Postal District
Property Type
No of Bedroom
Monthly Rent ($)
Floor Area (SQM)
Floor Area (SQFT)
Lease Commencement Date
district
Nanyang Primary School
Rosyth School
Henry Park Primary School
Tao Nan School
Raffles Girls' Primary School
St. Hilda's Primary School
Pei Hwa Presbyterian Primary School
Methodist Girls' School (Primary)
Nan Hua Primary School
Chij St. Nicholas Girls' School
Anglo-Chinese School (Primary)
Catholic High School (Primary)
Rulanh Primary School
Red Swastika School
Ai Tong School
St. Joseph's Institution Junior
Kong Hwa School
South View Primary School
Chongfu School
Pei Chun Public School
Holy Innocents' Primary School
Maris Stella High School (Primary)
Singapore Chinese Girls' Primary School
Canberra Primary School
Radin Mas Primary School
River Valley Primary School
Gongshang Primary School
Temasek Primary School
Anderson Primary School
Princess Elizabeth Primary School
Admiralty_MRT
Aljunied_

In [29]:
# df.head()

## Central Core Region

In [30]:
dff = df[df["Postal District"].isin([1, 2, 9, 10, 11])] if REGION == "ccr" else df
dff.tail()

,Project Name,onemap_address,latitude,longitude,Street Name,Postal District,Property Type,No of Bedroom,Monthly Rent ($),Floor Area (SQM),...,1st Post Offices dist,2nd Post Offices,2nd Post Offices dist,3rd Post Offices,3rd Post Offices dist,4th Post Offices,4th Post Offices dist,5th Post Offices,5th Post Offices dist,Project Name in Realis
1413643,THE GREENWICH,3 SELETAR ROAD THE GREENWICH SINGAPORE 807012,1.387188,103.869337,SELETAR ROAD,28,Non-landed Properties,NaN,3000,50 to 60,...,142 m,Post Box,965 m,Popstation @ The Seletar Mall,1.3 km,Post Box at Sengkang West Way,1.6 km,Singpost post box,2.2 km,THE GREENWICH
1413644,THE GREENWICH,3 SELETAR ROAD THE GREENWICH SINGAPORE 807012,1.387188,103.869337,SELETAR ROAD,28,Non-landed Properties,2.0,3200,80 to 90,...,142 m,Post Box,965 m,Popstation @ The Seletar Mall,1.3 km,Post Box at Sengkang West Way,1.6 km,Singpost post box,2.2 km,THE GREENWICH
1413645,THE TOPIARY,13 FERNVALE LANE THE TOPIARY SINGAPORE 797496,1.388816,103.872392,FERNVALE LANE,28,Executive Condominium,4.0,5000,120 to 130,...,415 m,Popstation @ The Seletar Mall,905 m,Post Box at Sengkang West Way,2.1 km,Post Box,1.0 km,Singpost post box,1.7 km,THE TOPIARY
1413646,THE TOPIARY,13 FERNVALE LANE THE TOPIARY SINGAPORE 797496,1.388816,103.872392,FERNVALE LANE,28,Executive Condominium,3.0,3600,80 to 90,...,415 m,Popstation @ The Seletar Mall,905 m,Post Box at Sengkang West Way,2.1 km,Post Box,1.0 km,Singpost post box,1.7 km,THE TOPIARY
1413647,TUAN SING PARK,1 UPPER NERAM ROAD TUAN SING PARK SINGAPORE 80...,1.387695,103.865330,DEDAP ROAD,28,Semi-Detached House,NaN,8000,250 to 300,...,237 m,Singpost Post Box,357 m,Post Box at Sengkang West Way,1.8 km,Popstation @ The Seletar Mall,1.6 km,Singpost post box,2.4 km,TUAN SING PARK


In [31]:
dff["Lease Commencement Date"] = pd.to_datetime(dff["Lease Commencement Date"], errors="coerce")
dff["year"] = dff["Lease Commencement Date"].dt.year
year_counts = dff["year"].value_counts().sort_index()
print(year_counts)

year
1999.0      150
2000.0    16778
2001.0    20528
2002.0    19873
2003.0    21759
2004.0    23717
2005.0    25890
2006.0    23395
2007.0    24331
2008.0    30938
2009.0    33716
2010.0    37869
2011.0    39768
2012.0    43800
2013.0    46695
2014.0    52284
2015.0    59171
2016.0    64991
2017.0    70392
2018.0    77134
2019.0    81259
2020.0    79752
2021.0    86025
2022.0    78805
2023.0    72315
2024.0    76316
2025.0    79468
2026.0     7385
Name: count, dtype: int64


C:\Users\cecel\AppData\Local\Temp\ipykernel_17000\2832372109.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["year"] = dff["Lease Commencement Date"].dt.year


### Rent per sqft

In [32]:
def sqft_median(x):
    if pd.isna(x):
        return None
    
    x = str(x).replace(",", "")  # remove commas
    
    if "to" in x:
        low, high = x.split("to")
        return (float(low) + float(high)) / 2
    else:
        return float(x)  # if single value
dff["FloorArea_median_sqft"] = dff["Floor Area (SQFT)"].apply(sqft_median)
dff["rent_per_sqft"] = dff["Monthly Rent ($)"] / dff["FloorArea_median_sqft"]

C:\Users\cecel\AppData\Local\Temp\ipykernel_17000\217578015.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["FloorArea_median_sqft"] = dff["Floor Area (SQFT)"].apply(sqft_median)
C:\Users\cecel\AppData\Local\Temp\ipykernel_17000\217578015.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["rent_per_sqft"] = dff["Monthly Rent ($)"] / dff["FloorArea_median_sqft"]


In [33]:
cols_to_drop = ["onemap_address", "Street Name", "Project Name in Realis", "district", "Floor Area (SQM)", "Monthly Rent ($)", "Floor Area (SQFT)", 
                "FloorArea_median_sqft", "Neighbourhood",
    "1st Bus Stops",    "2nd Bus Stops",    "3rd Bus Stops",    "4th Bus Stops",    "5th Bus Stops",
    "1st Supermarkets",    "2nd Supermarkets",    "3rd Supermarkets",    "4th Supermarkets",    "5th Supermarkets",
    "1st Parks",    "2nd Parks",    "3rd Parks",    "4th Parks",    "5th Parks",
    "1st Clinics",    "2nd Clinics",    "3rd Clinics",    "4th Clinics",    "5th Clinics",
    "1st Bank",    "2nd Bank",    "3rd Bank",    "4th Bank",    "5th Bank",
    "1st ATMs",    "2nd ATMs",    "3rd ATMs",    "4th ATMs",    "5th ATMs",
    "1st Post Boxes",    "2nd Post Boxes",    "3rd Post Boxes",    "4th Post Boxes",    "5th Post Boxes",
    "1st Post Offices",    "2nd Post Offices",    "3rd Post Offices",    "4th Post Offices",    "5th Post Offices",
]
dff = dff.drop(columns=cols_to_drop, errors="ignore")

categorical_cols = ["Postal District", "Property Type", 'Planning Area', 'Planning Region']
dff = pd.get_dummies(
    dff,
    columns=categorical_cols,
    prefix=categorical_cols,
    dummy_na=False,
    dtype=int
)

dff.tail()

,Project Name,latitude,longitude,No of Bedroom,Lease Commencement Date,Nanyang Primary School,Rosyth School,Henry Park Primary School,Tao Nan School,Raffles Girls' Primary School,...,Planning Area_Tanglin,Planning Area_Tengah,Planning Area_Toa Payoh,Planning Area_Woodlands,Planning Area_Yishun,Planning Region_Central Region,Planning Region_East Region,Planning Region_North East Region,Planning Region_North Region,Planning Region_West Region
1413643,THE GREENWICH,1.387188,103.869337,NaN,2026-01-01,NaN,1.717273,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0,0
1413644,THE GREENWICH,1.387188,103.869337,2.0,2026-01-01,NaN,1.717273,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0,0
1413645,THE TOPIARY,1.388816,103.872392,4.0,2026-01-01,NaN,1.797062,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0,0
1413646,THE TOPIARY,1.388816,103.872392,3.0,2026-01-01,NaN,1.797062,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0,0
1413647,TUAN SING PARK,1.387695,103.865330,NaN,2026-01-01,NaN,1.977431,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0,0


### Distance processing

In [34]:
dist_cols = [col for col in dff.columns if " dist" in col]
print(dist_cols)
mrt_cols = [col for col in dff.columns if "_MRT" in col]
print(mrt_cols)
scl_cols = [col for col in dff.columns if "School" in col or "Institution" in col]
print(len(scl_cols))

['1st Bus Stops dist', '2nd Bus Stops dist', '3rd Bus Stops dist', '4th Bus Stops dist', '5th Bus Stops dist', '1st Supermarkets dist', '2nd Supermarkets dist', '3rd Supermarkets dist', '4th Supermarkets dist', '5th Supermarkets dist', '1st Parks dist', '2nd Parks dist', '3rd Parks dist', '4th Parks dist', '5th Parks dist', '1st Clinics dist', '2nd Clinics dist', '3rd Clinics dist', '4th Clinics dist', '5th Clinics dist', '1st Bank dist', '2nd Bank dist', '3rd Bank dist', '4th Bank dist', '5th Bank dist', '1st ATMs dist', '2nd ATMs dist', '3rd ATMs dist', '4th ATMs dist', '5th ATMs dist', '1st Post Boxes dist', '2nd Post Boxes dist', '3rd Post Boxes dist', '4th Post Boxes dist', '5th Post Boxes dist', '1st Post Offices dist', '2nd Post Offices dist', '3rd Post Offices dist', '4th Post Offices dist', '5th Post Offices dist']
['Admiralty_MRT', 'Aljunied_MRT', 'Ang Mo Kio_MRT', 'Bangkit_MRT', 'Bartley_MRT', 'Bayfront_MRT', 'Bayshore_MRT', 'Beauty World_MRT', 'Bedok_MRT', 'Bedok North_MRT'

In [35]:
def convert_to_meters(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip().lower()
    
    # extract numeric value
    match = re.search(r"[\d\.]+", x)
    if not match:
        return None
    
    value = float(match.group())
    
    # convert based on unit
    if "km" in x:
        return value
    elif "m" in x:
        return value / 1000
    else:
        return None  # unknown unit
        
for col in dist_cols:
    dff[col] = dff[col].apply(convert_to_meters)
dff[mrt_cols] = dff[mrt_cols] / 1000

dff[dist_cols] = dff[dist_cols].fillna(5)
dff[mrt_cols] = dff[mrt_cols].fillna(5)
dff[scl_cols] = dff[scl_cols].fillna(3)

dff.tail()

,Project Name,latitude,longitude,No of Bedroom,Lease Commencement Date,Nanyang Primary School,Rosyth School,Henry Park Primary School,Tao Nan School,Raffles Girls' Primary School,...,Planning Area_Tanglin,Planning Area_Tengah,Planning Area_Toa Payoh,Planning Area_Woodlands,Planning Area_Yishun,Planning Region_Central Region,Planning Region_East Region,Planning Region_North East Region,Planning Region_North Region,Planning Region_West Region
1413643,THE GREENWICH,1.387188,103.869337,NaN,2026-01-01,3.0,1.717273,3.0,3.0,3.0,...,0,0,0,0,0,0,0,1,0,0
1413644,THE GREENWICH,1.387188,103.869337,2.0,2026-01-01,3.0,1.717273,3.0,3.0,3.0,...,0,0,0,0,0,0,0,1,0,0
1413645,THE TOPIARY,1.388816,103.872392,4.0,2026-01-01,3.0,1.797062,3.0,3.0,3.0,...,0,0,0,0,0,0,0,1,0,0
1413646,THE TOPIARY,1.388816,103.872392,3.0,2026-01-01,3.0,1.797062,3.0,3.0,3.0,...,0,0,0,0,0,0,0,1,0,0
1413647,TUAN SING PARK,1.387695,103.865330,NaN,2026-01-01,3.0,1.977431,3.0,3.0,3.0,...,0,0,0,0,0,0,0,1,0,0


### Missing value processing

In [36]:
has_cols = [col for col in dff.columns if "Has_" in col]
print(len(has_cols))
dff[has_cols] = dff[has_cols].fillna(0)

64


In [37]:
dff = dff.dropna(subset=["Lease Commencement Date"])
for col in ["Condo_Age_2026", "house_age_2026", "tenure_remaining_years"]:
    dff[col] = dff[col].fillna(dff[col].median())
dff["No of Bedroom_missing"] = dff["No of Bedroom"].isna().astype(int)
dff["Large_Dev_missing"] = dff["Large_Dev_200plus"].isna().astype(int)
fill_neg_cols = ["No of Bedroom"]
dff[fill_neg_cols] = dff[fill_neg_cols].fillna(-1)
fill_zero_cols = ["project_size_small", "project_size_medium", "project_size_large", "Large_Dev_200plus"]
dff[fill_zero_cols] = dff[fill_zero_cols].fillna(0)

cols = ["sora_overnight_pct", "sora_compounded_3m_pct", "ura_private_rental_index_yoy_pct", "ura_private_price_index_avg_3seg_yoy_pct"]
dff = dff.sort_values("Lease Commencement Date")
dff[cols] = dff[cols].ffill().bfill()


C:\Users\cecel\AppData\Local\Temp\ipykernel_17000\2840926910.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["No of Bedroom_missing"] = dff["No of Bedroom"].isna().astype(int)
C:\Users\cecel\AppData\Local\Temp\ipykernel_17000\2840926910.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["Large_Dev_missing"] = dff["Large_Dev_200plus"].isna().astype(int)


In [38]:
dff

,Project Name,latitude,longitude,No of Bedroom,Lease Commencement Date,Nanyang Primary School,Rosyth School,Henry Park Primary School,Tao Nan School,Raffles Girls' Primary School,...,Planning Area_Toa Payoh,Planning Area_Woodlands,Planning Area_Yishun,Planning Region_Central Region,Planning Region_East Region,Planning Region_North East Region,Planning Region_North Region,Planning Region_West Region,No of Bedroom_missing,Large_Dev_missing
114592,FAR EAST PLAZA,1.307177,103.833793,-1.0,1999-11-01,3.000000,3.000000,3.000000,3.000000,3.000000,...,0,0,0,0,0,0,0,0,1,1
89583,THE PLAZA,1.299620,103.860647,-1.0,1999-11-01,3.000000,3.000000,3.000000,3.000000,3.000000,...,0,0,0,1,0,0,0,0,1,1
70348,REGENT PARK,1.317260,103.761545,-1.0,1999-11-01,3.000000,3.000000,3.000000,3.000000,3.000000,...,0,0,0,0,0,0,0,1,1,0
339803,MANDARIN GARDENS,1.306700,103.925754,-1.0,1999-11-01,3.000000,3.000000,3.000000,1.669276,3.000000,...,0,0,0,0,1,0,0,0,1,0
339829,MANDARIN GARDENS,1.306700,103.925754,-1.0,1999-11-01,3.000000,3.000000,3.000000,1.669276,3.000000,...,0,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408718,D'LEEDON,1.316090,103.802300,1.0,2026-01-01,0.719757,3.000000,1.980323,3.000000,1.623122,...,0,0,0,1,0,0,0,0,0,0
1408717,D'LEEDON,1.316090,103.802300,3.0,2026-01-01,0.719757,3.000000,1.980323,3.000000,1.623122,...,0,0,0,1,0,0,0,0,0,0
1408716,D'LEEDON,1.316090,103.802300,3.0,2026-01-01,0.719757,3.000000,1.980323,3.000000,1.623122,...,0,0,0,1,0,0,0,0,0,0
1408714,D'LEEDON,1.316090,103.802300,2.0,2026-01-01,0.719757,3.000000,1.980323,3.000000,1.623122,...,0,0,0,1,0,0,0,0,0,0


In [39]:
count_le_2024 = (dff["year"] <= 2023).sum()
count_gt_2024 = (dff["year"] > 2023).sum()

print("<= 2023:", count_le_2024)
print("> 2023:", count_gt_2024)

<= 2023: 1131335
> 2023: 163169


### Tenure Clipped

In [40]:
# 1. Divide tenure by 10
dff["tenure_remaining_years_div100"] = dff["tenure_remaining_years"] / 100

# 2. Show describe statistics
print(dff["tenure_remaining_years_div100"].describe())

# 3. Clip values at 10 (i.e., max = 10)
dff["tenure_remaining_years_div100"] = dff["tenure_remaining_years_div100"].clip(upper=10)

# (Optional) check after clipping
print(dff["tenure_remaining_years_div100"].describe())
dff = dff.drop(columns=["tenure_remaining_years"])

count    1.294504e+06
mean     5.320625e+01
std      4.936933e+01
min      4.300000e-01
25%      9.900000e-01
50%      9.999000e+01
75%      9.999000e+01
max      9.999000e+01
Name: tenure_remaining_years_div100, dtype: float64
count    1.294504e+06
mean     5.798112e+00
std      4.493861e+00
min      4.300000e-01
25%      9.900000e-01
50%      1.000000e+01
75%      1.000000e+01
max      1.000000e+01
Name: tenure_remaining_years_div100, dtype: float64


C:\Users\cecel\AppData\Local\Temp\ipykernel_17000\3664585040.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["tenure_remaining_years_div100"] = dff["tenure_remaining_years"] / 100


### Reordering

In [41]:
def reorder_col(dff):
    print("Before column reordering: ", len(dff.columns.tolist()))
    # --- Base columns ---
    base_cols = [
        "Project Name",
        "Lease Commencement Date",
        "latitude",
        "longitude",
        "No of Bedroom",
        "No of Bedroom_missing",
    ]
    
    # --- Structural + Macro ---
    struct_macro_cols = [
        col for col in dff.columns
        if any(x in col for x in [
            "Large_Dev", "_Age_", "_age_",
            "tenure", "cpi", "unemployment",
            "sora", "bond", "sgd", "ura",
            "hdb", "gva"
        ])
    ]
    
    # --- One-hot ---
    onehot_cols = [
        col for col in dff.columns
        if any(x in col for x in [
            "Postal District",
            "Property Type",
            "Planning Area",
            "Planning Region",
            "project_size"
        ])
    ]
    
    # --- Schools ---
    school_cols = [
        col for col in dff.columns
        if "School" in col or "Institution" in col
    ]
    
    # --- MRT ---
    mrt_cols = [
        col for col in dff.columns
        if "_MRT" in col
    ]
    
    # --- Distances ---
    dist_cols = [
        col for col in dff.columns
        if "dist" in col.lower()
    ]
    
    # --- Facilities ---
    facility_cols = [
        col for col in dff.columns
        if col.startswith("Has_")
    ]
    
    # --- Engineered ---
    engineered_cols = [
        col for col in dff.columns
        if col in ["year", "rent_per_sqft"] or col.endswith("_missing")
    ]
    
    # --- Combine in order ---
    ordered_cols = (
        base_cols
        + struct_macro_cols
        + onehot_cols
        + school_cols
        + mrt_cols
        + dist_cols
        + facility_cols
        + engineered_cols
    )
    
    # --- Remove duplicates while preserving order ---
    ordered_cols = list(dict.fromkeys(ordered_cols))
    
    # --- Add any remaining columns (safety) ---
    remaining_cols = [col for col in dff.columns if col not in ordered_cols]
    ordered_cols += remaining_cols
    
    # --- Reorder dataframe ---
    dff = dff[ordered_cols]
    print("After column reordering: ", len(dff.columns.tolist()))
    return dff
dff = reorder_col(dff)

Before column reordering:  397
After column reordering:  397


In [42]:
# dff.to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v1.csv")
dff.describe().to_csv(rf"dataset\URA_merged_{REGION}_v1_describe.csv")
dff.describe()

,Lease Commencement Date,latitude,longitude,No of Bedroom,No of Bedroom_missing,Large_Dev_200plus,Condo_Age_2026,tenure_freehold_like,tenure_medium_lease_60_80,tenure_more_than_80,...,Has_Underwater Fitness Station,Has_Viewing Deck,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft
count,1294504,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,...,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06,1.294504e+06
mean,2016-06-22 01:06:25.200200,1.323776e+00,1.038464e+02,1.194537e+00,3.413925e-01,4.923384e-01,2.891583e+01,5.008698e-01,3.365768e-03,4.656795e-01,...,2.772104e-02,4.087975e-02,5.008922e-01,7.860153e-03,1.327157e-01,4.341122e-02,2.069519e-03,6.098629e-02,2.016035e+03,3.682731e+00
min,1999-11-01 00:00:00,1.239785e+00,1.036918e+02,-1.000000e+00,0.000000e+00,0.000000e+00,3.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.999000e+03,4.222222e-01
25%,2012-02-01 00:00:00,1.299620e+00,1.038120e+02,-1.000000e+00,0.000000e+00,0.000000e+00,2.800000e+01,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.012000e+03,2.545455e+00
50%,2017-11-01 00:00:00,1.315541e+00,1.038420e+02,2.000000e+00,0.000000e+00,0.000000e+00,2.800000e+01,1.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.017000e+03,3.435294e+00
75%,2021-11-01 00:00:00,1.342915e+00,1.038845e+02,3.000000e+00,1.000000e+00,1.000000e+00,2.800000e+01,1.000000e+00,0.000000e+00,1.000000e+00,...,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.021000e+03,4.606061e+00
max,2026-01-01 00:00:00,1.461673e+00,1.039734e+02,8.000000e+00,1.000000e+00,1.000000e+00,6.000000e+01,1.000000e+00,1.000000e+00,1.000000e+00,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2.026000e+03,2.384615e+01
std,NaN,3.414222e-02,5.685261e-02,1.744271e+00,4.741770e-01,4.999415e-01,5.432240e+00,4.999994e-01,5.791755e-02,4.988209e-01,...,1.641725e-01,1.980117e-01,4.999994e-01,8.830842e-02,3.392673e-01,2.037811e-01,4.544488e-02,2.393053e-01,6.678754e+00,1.600824e+00


In [43]:
dfg = (
    dff.groupby(["Project Name", "Lease Commencement Date"])
      .mean(numeric_only=True)
      .reset_index()
)
dfg = reorder_col(dfg)
# dfg.to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v2.csv")
dfg.describe().to_csv(rf"dataset\URA_merged_{REGION}_v2_describe.csv")
dfg.describe()

C:\Users\cecel\AppData\Local\Temp\ipykernel_17000\3842093882.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


Before column reordering:  397
After column reordering:  397


,Lease Commencement Date,latitude,longitude,No of Bedroom,No of Bedroom_missing,Large_Dev_200plus,Condo_Age_2026,tenure_freehold_like,tenure_medium_lease_60_80,tenure_more_than_80,...,Has_Underwater Fitness Station,Has_Viewing Deck,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft
count,321701,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,...,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000,321701.000000
mean,2015-09-08 07:45:16.486426,1.324724,103.848718,0.952190,0.431621,0.220379,29.419831,0.686426,0.003447,0.252405,...,0.009829,0.015847,0.390807,0.003174,0.088750,0.019540,0.001368,0.029577,2015.245672,3.320846
min,1999-11-01 00:00:00,1.239785,103.691841,-1.000000,0.000000,0.000000,3.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1999.000000,0.428571
25%,2010-08-01 00:00:00,1.305834,103.822962,-1.000000,0.000000,0.000000,28.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2010.000000,2.288141
50%,2016-11-01 00:00:00,1.315854,103.842145,1.333333,0.000000,0.000000,28.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2016.000000,3.116705
75%,2021-07-01 00:00:00,1.339956,103.885941,2.666667,1.000000,0.000000,29.000000,1.000000,0.000000,1.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2021.000000,4.153846
max,2026-01-01 00:00:00,1.461673,103.973377,8.000000,1.000000,1.000000,60.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2026.000000,23.846154
std,NaN,0.031522,0.052255,1.786429,0.490193,0.414503,6.447768,0.463946,0.058613,0.434393,...,0.098653,0.124884,0.487932,0.056247,0.284383,0.138413,0.036958,0.169418,6.945820,1.467980


In [44]:
dfg

,Project Name,Lease Commencement Date,latitude,longitude,No of Bedroom,No of Bedroom_missing,Large_Dev_200plus,Condo_Age_2026,tenure_freehold_like,tenure_medium_lease_60_80,...,Has_Underwater Fitness Station,Has_Viewing Deck,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft
0,# 1 LOFT,2016-04-01,1.312763,103.883519,0.384615,0.615385,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,3.887744
1,# 1 LOFT,2016-05-01,1.312763,103.883519,1.200000,0.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,4.044225
2,# 1 LOFT,2016-06-01,1.312763,103.883519,1.400000,0.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,4.265359
3,# 1 LOFT,2016-07-01,1.312763,103.883519,1.000000,0.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,4.333333
4,# 1 LOFT,2016-08-01,1.312763,103.883519,1.000000,0.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,4.111111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321696,ZEPHYR PARK,2023-11-01,1.338309,103.952206,-1.000000,1.000000,0.0,33.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023.0,3.428571
321697,ZEPHYR PARK,2024-10-01,1.338309,103.952206,-1.000000,1.000000,0.0,33.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024.0,3.314286
321698,ZYANYA,2025-08-01,1.313921,103.883276,-1.000000,1.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,4.065934
321699,ZYANYA,2025-10-01,1.313921,103.883276,1.333333,0.333333,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,5.577534


In [45]:
count_le_2024 = (dfg["year"] <= 2023).sum()
count_gt_2024 = (dfg["year"] > 2023).sum()

print("<= 2023:", count_le_2024)
print("> 2023:", count_gt_2024)

<= 2023: 284546
> 2023: 37155


In [46]:
# cols = [col for col in df.columns if col != "Lease Commencement Date"]

# constant_by_date_cols = []

# for col in cols:
#     nunique_per_date = df.groupby("Lease Commencement Date")[col].nunique(dropna=False)
    
#     if (nunique_per_date <= 1).all():
#         constant_by_date_cols.append(col)

# print("Columns constant within each date:")
# print(constant_by_date_cols)

### Macro data

In [47]:
import pandas as pd

constant_by_date_cols = [
    "cpi_all_items_infl_yoy_pct",
    "unemployment_rate_sa_pct",
    "sora_overnight_pct",
    "sora_compounded_3m_pct",
    "sg_govt_bond_yield_10y_pct",
    "sgd_per_usd_logret_12m_pct",
    "ura_private_rental_index_yoy_pct",
    "ura_private_price_index_avg_3seg_yoy_pct",
    "hdb_resale_price_index_yoy_pct",
    "gva_yoy_growth_pct",
]

target_col = "rent_per_sqft"
group_cols = ["Project Name", "Lease Commencement Date"]

dfg["Lease Commencement Date"] = pd.to_datetime(
    dfg["Lease Commencement Date"], errors="coerce"
)
dfg = dfg.sort_values(["Project Name", "Lease Commencement Date"]).copy()

project_level_cols = [
    col for col in dfg.columns
    if col not in group_cols + constant_by_date_cols + [target_col]
]

# build date-level macro table
macro_by_date = (
    dfg.groupby("Lease Commencement Date")[constant_by_date_cols]
       .first()
       .sort_index()
)

# fill missing months in macro table too
full_macro_dates = pd.date_range(
    start=dfg["Lease Commencement Date"].min(),
    end=dfg["Lease Commencement Date"].max(),
    freq="MS"
)

macro_by_date = macro_by_date.reindex(full_macro_dates)
macro_by_date.index.name = "Lease Commencement Date"
macro_by_date = macro_by_date.interpolate(method="time", limit_direction="both")


def expand_project(project_name, group):
    group = group.sort_values("Lease Commencement Date").copy()

    full_dates = pd.date_range(
        start=group["Lease Commencement Date"].min(),
        end=group["Lease Commencement Date"].max(),
        freq="MS"
    )

    g = (
        group.set_index("Lease Commencement Date")
             .reindex(full_dates)
             .rename_axis("Lease Commencement Date")
             .reset_index()
    )

    g["Project Name"] = project_name

    # ✅ STEP 1: mark original vs missing BEFORE interpolation
    g["y_mask"] = g["rent_per_sqft"].notna().astype(int)

    # fill project-level columns
    existing_project_cols = [c for c in project_level_cols if c in g.columns]
    g[existing_project_cols] = g[existing_project_cols].ffill().bfill()

    # ✅ STEP 2: interpolate rent
    g["rent_per_sqft"] = g["rent_per_sqft"].interpolate(
        method="linear",
        limit_direction="both"
    )

    # merge macro columns
    g = g.drop(columns=constant_by_date_cols, errors="ignore")

    g = g.merge(
        macro_by_date.reset_index(),
        on="Lease Commencement Date",
        how="left"
    )

    return g

expanded_list = []

for project_name, group in dfg.groupby("Project Name"):
    expanded_list.append(expand_project(project_name, group))

dfge = pd.concat(expanded_list, ignore_index=True)
dfge = reorder_col(dfge)
print(dfge.shape)
print(dfge.head())

Before column reordering:  398
After column reordering:  398
(670262, 398)
  Project Name Lease Commencement Date  latitude   longitude  No of Bedroom  \
0     # 1 LOFT              2016-04-01  1.312763  103.883519       0.384615   
1     # 1 LOFT              2016-05-01  1.312763  103.883519       1.200000   
2     # 1 LOFT              2016-06-01  1.312763  103.883519       1.400000   
3     # 1 LOFT              2016-07-01  1.312763  103.883519       1.000000   
4     # 1 LOFT              2016-08-01  1.312763  103.883519       1.000000   

   No of Bedroom_missing  Large_Dev_200plus  Condo_Age_2026  \
0               0.615385                0.0            28.0   
1               0.000000                0.0            28.0   
2               0.000000                0.0            28.0   
3               0.000000                0.0            28.0   
4               0.000000                0.0            28.0   

   tenure_freehold_like  tenure_medium_lease_60_80  ...  Has_Viewing De

In [48]:
dfge.describe().to_csv(rf"dataset\URA_merged_{REGION}_v3.0_describe.csv")
dfge.describe()

,Lease Commencement Date,latitude,longitude,No of Bedroom,No of Bedroom_missing,Large_Dev_200plus,Condo_Age_2026,tenure_freehold_like,tenure_medium_lease_60_80,tenure_more_than_80,...,Has_Viewing Deck,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft,y_mask
count,670262,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000,670262.00000,...,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000,670262.000000
mean,2015-01-04 22:18:41.493385,1.326080,103.854002,0.527814,0.562717,0.121527,30.111999,0.723378,0.002318,0.17378,...,0.009073,0.276708,0.002227,0.057685,0.011854,0.000755,0.018512,2014.212891,2.979939,0.479963
min,1999-11-01 00:00:00,1.239785,103.691841,-1.000000,0.000000,0.000000,3.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1999.000000,0.428571,0.000000
25%,2009-08-01 00:00:00,1.308890,103.825216,-1.000000,0.000000,0.000000,28.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2009.000000,1.987128,0.000000
50%,2015-11-01 00:00:00,1.316682,103.848059,-1.000000,1.000000,0.000000,28.000000,1.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2015.000000,2.764706,0.000000
75%,2020-12-01 00:00:00,1.341897,103.891387,2.400000,1.000000,0.000000,32.000000,1.000000,0.000000,0.00000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2020.000000,3.743457,1.000000
max,2026-01-01 00:00:00,1.461673,103.973377,8.000000,1.000000,1.000000,60.000000,1.000000,1.000000,1.00000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2026.000000,23.846154,1.000000
std,NaN,0.030697,0.051770,1.803736,0.493061,0.326739,6.970666,0.447328,0.048095,0.37892,...,0.094817,0.447371,0.047144,0.233147,0.108227,0.027466,0.134794,7.120713,1.418841,0.499599


In [49]:
import numpy as np
# =========================
# Config
# =========================
timestamp_col = "Lease Commencement Date"
project_col = "Project Name"
folder = rf"dataset\timestep_{REGION}"
coverage_check_path = rf"dataset/URA_merged_{REGION}_project_coverage_check.csv"

os.makedirs(folder, exist_ok=True)

# work on a copy
df_work = dfge.copy()

# =========================
# 1. Ensure timestamp is datetime
# =========================
df_work[timestamp_col] = pd.to_datetime(df_work[timestamp_col])

# optional: drop rows with missing key fields
df_work = df_work.dropna(subset=[timestamp_col, project_col])

# sort for consistency
df_work = df_work.sort_values([timestamp_col, project_col]).reset_index(drop=True)

# =========================
# 2. Define fixed project order
# =========================
project_order = sorted(df_work[project_col].unique())

df_work[project_col] = pd.Categorical(
    df_work[project_col],
    categories=project_order,
    ordered=True
)

# =========================
# 3. All timesteps
# =========================
all_ts = sorted(df_work[timestamp_col].dropna().unique())

# =========================
# 4. Build coverage check
# =========================
records = []
for project, g in df_work.groupby(project_col, observed=False):
    ts_present = set(g[timestamp_col])
    start = g[timestamp_col].min()
    end = g[timestamp_col].max()

    active_range = [ts for ts in all_ts if start <= ts <= end]
    missing_within_period = [ts for ts in active_range if ts not in ts_present]

    records.append({
        "project_name": project,
        "earliest_timestep": start,
        "latest_timestep": end,
        "num_records": len(g),
        "expected_timesteps_in_active_period": len(active_range),
        "missing_timesteps_within_period": len(missing_within_period),
    })

coverage_check = pd.DataFrame(records)
coverage_check["earliest_timestep"] = pd.to_datetime(
    coverage_check["earliest_timestep"]
).dt.strftime("%Y-%m-%d")
coverage_check["latest_timestep"] = pd.to_datetime(
    coverage_check["latest_timestep"]
).dt.strftime("%Y-%m-%d")

coverage_check.to_csv(coverage_check_path, index=False)
print(f"Saved: {coverage_check_path}")

# =========================
# 5. Prepare columns for zero filling
# =========================
# columns not to zero-fill
exclude_cols = [timestamp_col, project_col, "year"]

# keep columns that actually exist
exclude_cols = [c for c in exclude_cols if c in df_work.columns]

feature_cols = [c for c in df_work.columns if c not in exclude_cols]

numeric_cols = df_work[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in feature_cols if c not in numeric_cols]

# =========================
# 6. Remove duplicates if needed
#    If same (timestamp, project) appears multiple times,
#    keep first row. Change this if you prefer mean/sum.
# =========================
df_work = df_work.drop_duplicates(subset=[timestamp_col, project_col], keep="first")

# lookup table
lookup = df_work.set_index([timestamp_col, project_col])

# =========================
# 7. Rewrite each timestep CSV
#    Missing project rows -> zeros
# =========================
file_lengths = []

for ts in all_ts:
    rows = []

    for project in project_order:
        row = {
            timestamp_col: ts,
            project_col: project,
        }

        key = (ts, project)

        if key in lookup.index:
            values = lookup.loc[key]

            for col in numeric_cols:
                row[col] = values[col]

            for col in non_numeric_cols:
                row[col] = values[col]
        else:
            # missing project at this timestep -> fill zeros
            for col in numeric_cols:
                row[col] = 0

            for col in non_numeric_cols:
                row[col] = 0

        rows.append(row)

    out_df = pd.DataFrame(rows)

    # enforce fixed order
    out_df[project_col] = pd.Categorical(
        out_df[project_col],
        categories=project_order,
        ordered=True
    )
    out_df = out_df.sort_values(project_col).reset_index(drop=True)

    filename = rf"{folder}\data_{pd.Timestamp(ts).strftime('%Y%m%d')}.csv"
    out_df.to_csv(filename, index=False)

    file_lengths.append(len(out_df))

# =========================
# 8. Final length check
# =========================
lengths_series = pd.Series(file_lengths)

if lengths_series.nunique() == 1:
    print(f"✅ All timesteps have consistent length: {lengths_series.iloc[0]}")
else:
    print("❌ Inconsistent lengths across timesteps!")
    print(lengths_series.value_counts())

Saved: dataset/URA_merged_full_project_coverage_check.csv
✅ All timesteps have consistent length: 3234


In [52]:
lat_col = "latitude"
lon_col = "longitude"
project_col = "Project Name"

# 1. Aggregate lat/lon per project
project_geo = (
    dfge.groupby(project_col)[[lat_col, lon_col]]
    .mean()   # robust if slight variations exist
    .reset_index()
)

# 2. Merge into coverage_check
coverage_check = coverage_check.merge(
    project_geo,
    left_on="project_name",
    right_on=project_col,
    how="left"
)

# 3. Drop duplicate column
coverage_check = coverage_check.drop(columns=[project_col])

# 4. Save
coverage_check.to_csv(coverage_check_path, index=False)

print("✅ coverage_check updated with latitude & longitude")

KeyError: 'Project Name'

In [55]:
dfge

,Project Name,Lease Commencement Date,latitude,longitude,No of Bedroom,No of Bedroom_missing,Large_Dev_200plus,Condo_Age_2026,tenure_freehold_like,tenure_medium_lease_60_80,...,Has_Viewing Deck,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft,y_mask
0,# 1 LOFT,2016-04-01,1.312763,103.883519,0.384615,0.615385,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,3.887744,1
1,# 1 LOFT,2016-05-01,1.312763,103.883519,1.200000,0.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,4.044225,1
2,# 1 LOFT,2016-06-01,1.312763,103.883519,1.400000,0.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,4.265359,1
3,# 1 LOFT,2016-07-01,1.312763,103.883519,1.000000,0.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,4.333333,1
4,# 1 LOFT,2016-08-01,1.312763,103.883519,1.000000,0.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0,4.111111,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
670257,ZEPHYR PARK,2024-10-01,1.338309,103.952206,-1.000000,1.000000,0.0,33.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024.0,3.314286,1
670258,ZYANYA,2025-08-01,1.313921,103.883276,-1.000000,1.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,4.065934,1
670259,ZYANYA,2025-09-01,1.313921,103.883276,-1.000000,1.000000,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,4.821734,0
670260,ZYANYA,2025-10-01,1.313921,103.883276,1.333333,0.333333,0.0,28.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,5.577534,1


In [54]:
df_ccr = pd.read_csv("dataset/URA_merged_ccr_project_coverage_check.csv")
df_full = pd.read_csv("dataset/URA_merged_full_project_coverage_check.csv")
project_col_ccr = "project_name"
project_col_full = "project_name"
lat_col = "latitude"
lon_col = "longitude"
# 1. build project -> lat/lon mapping from full
project_geo = (
    df_full.groupby(project_col_full)[[lat_col, lon_col]]
    .mean()
    .reset_index()
)

# 2. remove old lat/lon in ccr first if already exists
cols_to_drop = [c for c in [lat_col, lon_col] if c in df_ccr.columns]
if cols_to_drop:
    df_ccr = df_ccr.drop(columns=cols_to_drop)

# 3. merge
df_ccr = df_ccr.merge(
    project_geo,
    left_on=project_col_ccr,
    right_on=project_col_full,
    how="left"
)

# 4. if duplicate project_name column appears, drop one
if project_col_ccr != project_col_full and project_col_full in df_ccr.columns:
    df_ccr = df_ccr.drop(columns=[project_col_full])

# 5. save back
df_ccr.to_csv("dataset/URA_merged_ccr_project_coverage_check.csv", index=False)

print("Done")
print("Missing Latitude:", df_ccr[lat_col].isna().sum())
print("Missing Longitude:", df_ccr[lon_col].isna().sum())

Done
Missing Latitude: 0
Missing Longitude: 0


In [322]:
dfge["Lease Commencement Date"] = pd.to_datetime(
    dfge["Lease Commencement Date"], errors="coerce"
)

dfgep = dfge.sort_values(["Project Name", "Lease Commencement Date"]).copy()

# add previous timestep rent_per_sqft
dfgep["prev_y"] = dfgep.groupby("Project Name")["rent_per_sqft"].shift(1)

# remove earliest record of each project (where prev_y is NaN)
dfgep = dfgep[dfgep["prev_y"].notna()].reset_index(drop=True)
dfgep.to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v3.1.csv")
dfgep.describe().to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v3.1_describe.csv")
dfgep.describe()

,latitude,longitude,No of Bedroom,No of Bedroom_missing,Lease Commencement Date,Large_Dev_200plus,Condo_Age_2026,tenure_remaining_years,tenure_freehold_like,tenure_medium_lease_60_80,...,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft,y_mask,prev_y
count,204851.000000,204851.000000,204851.000000,204851.000000,204851,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,...,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000
mean,1.310452,103.826030,0.702434,0.531243,2014-10-07 03:31:53.833957,0.069704,30.886171,9032.887411,0.790955,0.004369,...,0.258402,0.002968,0.061015,0.012180,0.002050,0.018955,2014.099072,3.649956,0.533622,3.645505
min,1.272156,103.721044,-1.000000,0.000000,1999-12-01 00:00:00,0.000000,7.000000,43.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1999.000000,0.461538,0.000000,0.461538
25%,1.301067,103.813314,-1.000000,0.000000,2009-03-01 00:00:00,0.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2008.000000,2.590476,0.000000,2.586667
50%,1.312907,103.830428,-1.000000,1.000000,2015-08-01 00:00:00,0.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2015.000000,3.430454,1.000000,3.428312
75%,1.318783,103.839257,2.666667,1.000000,2020-11-01 00:00:00,0.000000,32.000000,9999.000000,1.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2020.000000,4.494422,1.000000,4.488325
max,1.402715,103.944841,8.000000,1.000000,2026-01-01 00:00:00,1.000000,58.000000,9999.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2026.000000,23.846154,1.000000,23.846154
std,0.012853,0.018234,1.897298,0.495708,NaN,0.254649,7.297005,2932.471296,0.406627,0.065954,...,0.437757,0.054399,0.239359,0.109687,0.045234,0.136367,7.181894,1.528155,0.498869,1.527576


In [9]:
count_le_2024 = (dfgep["year"] <= 2023).sum()
count_gt_2024 = (dfgep["year"] > 2023).sum()

print("<= 2023:", count_le_2024)
print("> 2023:", count_gt_2024)

<= 2023: 186445
> 2023: 18406
